In [0]:
# Imports

from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window
import dateutil.parser
import requests
import pandas as pd

In [0]:
# Funções de verificação

dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
# Preparação dos endereços das tabelas

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

print(catalog)
print(bronze_schema)
print(silver_schema)
print(gold_schema)
print(landing_path)

## Info Filmes

In [0]:
# Mapeamento tb_movies_info -> tb_info_filmes
silver_table_name = "tb_info_filmes"
df_info_filmes = spark.table(f"{bronze_schema}.tb_movies_info")

df_info_filmes = (
    df_info_filmes
        .where(F.col("id").isNotNull())
        .withColumnsRenamed({
            "id": "id_filme",
            "title": "titulo",
            "original_title": "titulo_original",
            "release_date": "data_lancamento",
            "runtime": "duracao_minutos",
            "original_language": "idioma_original",
            "status": "status_filme",
            "overview": "sinopse",
            "tagline": "frase_divulgacao"
        })
)


In [0]:
# Limpeza de status
df_info_filmes = (
    df_info_filmes.withColumn("status_filme",
                              F.upper(F.trim(F.regexp_replace(F.col("status_filme"), "-", " "))))
)

re_verify_status = "^(RELEASED|POST PRODUCTION|IN PRODUCTION|RUMORED|PLANNED|CANCELED)$"
standardized_condition = F.regexp(F.col("status_filme"), F.lit(re_verify_status))

dq_check(f"{silver_schema}.{silver_table_name}", "Verificação de status corrompidos", df_info_filmes, standardized_condition)

In [0]:
df_corrupted_status = (
    df_info_filmes
        .where(~standardized_condition)
        .select("status_filme")
)

# Verificação manual dos status corrompidos
display(df_corrupted_status)

In [0]:
# Tradução dos termos dos status para o português
df_info_filmes = (
    df_info_filmes.withColumn("status_filme", 
        F.when(F.col("status_filme") == "RELEASED", "Lançado")
            .when(F.col("status_filme") == "POST PRODUCTION", "Pós-Produção")
            .when(F.col("status_filme") == "IN PRODUCTION", "Em Produção")
            .when(F.col("status_filme") == "RUMORED", "Rumores")
            .when(F.col("status_filme") == "PLANNED", "Planejado")
            .when(F.col("status_filme") == "CANCELED", "Cancelado")
            .otherwise("Não Informado")
    )
)

df_distinct_status = (
    df_info_filmes
        .select("status_filme")
        .distinct()
)

dq_check(
    f"df_distinct_status", 
    "Verificação de status corrompidos após tratamento", 
    df_distinct_status, 
    F.col("status_filme").isin(
    ["Lançado", "Pós-Produção", "Em Produção", 
    "Rumores", "Planejado", "Cancelado", "Não Informado"]
    )
)

In [0]:
# Deduplicação de dados com base no ingestion_timestamp

ingestion_window = Window.partitionBy("id_filme").orderBy(F.desc("ingestion_datetime"))

df_info_filmes = (
    df_info_filmes
        .withColumn("row_number", F.row_number().over(ingestion_window))
        .where(F.col("row_number") == 1)
        .drop("row_number")
)

dq_check_unique(f"{silver_schema}.{silver_table_name}", "Verificação de duplicidade", df_info_filmes, ["id_filme"])

In [0]:
# Função para conversão em data a partir de lista de formatos
def to_date_formats(col, formats):
    return F.coalesce(*[F.try_to_date(col, f) for f in formats])

# Verificação de formatos de data
formats = ["dd/MM/yyyy", "dd-MM-yyyy", "MM/dd/yyyy", "MM-dd-yyyy", "yyyy/MM/dd", "yyyy-MM-dd"]
date_condition = to_date_formats(F.col("data_lancamento"), formats).isNotNull()
dq_check(f"{silver_schema}.{silver_table_name}", "Verificação de formatos de data", df_info_filmes, date_condition)

df_corrupted_date = (
    df_info_filmes.select("data_lancamento")
        .where(~date_condition)
)

# Verificação manual dos registros não comportados
display(df_corrupted_date)

In [0]:
df_info_filmes = (
    df_info_filmes.withColumn("data_lancamento", 
        to_date_formats(F.col("data_lancamento"), formats))
)

In [0]:
# Verificação de tipos

display(df_info_filmes.dtypes)

In [0]:
# Verificação de registros não numéricos

minute_type_check_condition = F.regexp(F.col("duracao_minutos"), F.lit("^[0-9]+$"))

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de tipos", 
    df_info_filmes,
    minute_type_check_condition)

display(
    df_info_filmes.select("duracao_minutos")
        .where(~minute_type_check_condition)
)

In [0]:
# Conversão de duração_minutos para inteiro

df_info_filmes = (
    df_info_filmes.withColumn("duracao_minutos", F.col("duracao_minutos").try_cast("int"))
)

dq_check(f"{silver_schema}.{silver_table_name}", 
    "Verificação de nulos", 
    df_info_filmes,
    F.col("duracao_minutos").isNotNull())

In [0]:
# Criação de coluna derivada e persistência da tabela na camada silver

df_info_filmes = (
    df_info_filmes.withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

df_info_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Financeiro Filmes

In [0]:
# Mapeamento tb_movies_financials -> tb_financeiro_filmes
silver_table_name = "tb_financeiro_filmes"
df_financeiro_filmes = spark.table(f"{bronze_schema}.tb_movies_financials")

df_financeiro_filmes = (
    df_financeiro_filmes
        .where(F.col("id").isNotNull())
        .dropDuplicates(["id"])
        .withColumnsRenamed({
            "budget": "orcamento_usd",
            "revenue": "receita_usd",
            "id": "id_filme"
        })
)

display(df_financeiro_filmes.limit(5))

In [0]:
# Verificação de tipos

display(df_financeiro_filmes.dtypes)

In [0]:
def isNumerical(column):
    return F.regexp(F.col(column), F.lit(r"[\d\.]+"))

dq_check(f"{silver_schema}.{silver_table_name}", 
    "Verificação de orcamentos não numéricos", 
    df_financeiro_filmes,
    isNumerical("orcamento_usd")
)

dq_check(f"{silver_schema}.{silver_table_name}", 
    "Verificação de receitas não numéricas", 
    df_financeiro_filmes,
    isNumerical("receita_usd")
)

In [0]:
# parse_money trata valores monetários (substitui cleans_numerical da última entrega, melhorado por IA)
def parse_money(col):
    s = F.upper(F.trim(F.col(col)))
    s = F.regexp_replace(s, r"USD|[\$\s,]", "")
    pat = r"^(-?\d+(?:\.\d+)?)([KMB]?)$"
    num, suf = F.regexp_extract(s, pat, 1), F.regexp_extract(s, pat, 2)
    mult = F.when(suf == "K", 1e3).when(suf == "M", 1e6).when(suf == "B", 1e9).otherwise(1)
    val = F.when(num != "", num.try_cast("decimal(30,4)") * mult)
    return F.when(val > 0, val).cast("decimal(18,2)")

df_financeiro_filmes = (
    df_financeiro_filmes
        .withColumn("orcamento_usd", parse_money("orcamento_usd"))
        .withColumn("receita_usd", parse_money("receita_usd"))
)

df_financeiro_filmes = (
    df_financeiro_filmes
        .withColumn("orcamento_usd",
            F.when(F.col("orcamento_usd") <= 0, None)
                .otherwise(F.col("orcamento_usd")))
        .withColumn("receita_usd",
            F.when(F.col("receita_usd") <= 0, None)
                .otherwise(F.col("receita_usd")))
)

display(df_financeiro_filmes.limit(5))

In [0]:
df_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

dollar_today = (
    df_cotacao_dolar
        .orderBy(F.col("dataHoraCotacao").desc())
        .limit(1)
        .select("cotacaoVenda")
        .collect()[0][0]
)

df_financeiro_filmes = (
    df_financeiro_filmes.withColumns({
        "orcamento_brl": (F.col("orcamento_usd") * dollar_today).cast("decimal(18,2)"),
        "receita_brl": (F.col("receita_usd") * dollar_today).cast("decimal(18,2)")
    })
)

display(df_financeiro_filmes.limit(5))

In [0]:
df_financeiro_filmes = (
    df_financeiro_filmes.withColumns(
        {
            "lucro_usd": F.try_subtract(F.col("receita_usd"), F.col("orcamento_usd")),
            "lucro_brl": F.try_subtract(F.col("receita_brl"), F.col("orcamento_brl")),
            "margem_lucro": F.try_divide(F.col("lucro_brl"), F.col("orcamento_brl")) * 100
        }
    )
)

display(df_financeiro_filmes.limit(5))

In [0]:
df_financeiro_filmes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Metricas Engajamento

In [0]:
# Mapeamento tb_movies_metrics -> tb_metricas_engajamento
silver_table_name = "tb_metricas_engajamento"
df_metricas_engajamento = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_metricas_engajamento = (
    df_metricas_engajamento
        .where(F.col("id").isNotNull())
        .dropDuplicates(["id"])
        .withColumnsRenamed({
            "id": "id_filme",
            "popularity": "popularidade",
            "vote_average": "nota_media_tmdb",
            "vote_count": "qtd_votos_tmdb",
            "averageRating": "nota_media_imdb",
            "numVotes": "qtd_votos_imdb"
        })
)

display(df_metricas_engajamento.limit(5))

In [0]:
# Limpeza de dados de popularidade
year_like_popularity = F.col("popularidade").rlike(r"^(19|20|21)\d{2}$")

popularity_type_check = F.regexp(
    F.col("popularidade"), 
    F.lit(r"^(\d+\.{0,1}\d+|\d+\,{0,1}\d+|\d+)$")
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de popularidades não numéricas", 
    df_metricas_engajamento,
    popularity_type_check
)

df_corrupted_popularity = df_metricas_engajamento.filter(~popularity_type_check)

display(df_corrupted_popularity.limit(5))

In [0]:
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("popularidade",
                    F.when(
                        popularity_type_check & ~year_like_popularity, 
                        F.regexp_replace(
                            F.col("popularidade"),
                            r"\,",
                            "."
                        ).cast("double")
                    ).otherwise(None)
        ).withColumn("popularidade",
            F.when(F.col("popularidade") <= 0, None)
                .otherwise(F.col("popularidade"))
        )
)

display(df_metricas_engajamento.limit(5))

In [0]:
# Limpeza de dados de notas médias

# Substituição de vírgula por ponto
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("nota_media_tmdb",
            F.regexp_replace(F.col("nota_media_tmdb"), r"\,", ".")
                .try_cast("double")
        ).withColumn("nota_media_imdb",
            F.regexp_replace(F.col("nota_media_imdb"), r"\,", ".")
                .try_cast("double")
        )
)

# Correção de limites de notas
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("nota_media_tmdb",
            F.when(F.col("nota_media_tmdb") > 10, None)
                .when(F.col("nota_media_tmdb") < 0, None)
                .otherwise(F.col("nota_media_tmdb"))
        ).withColumn("nota_media_imdb",
            F.when(F.col("nota_media_imdb") > 10, None)
                .when(F.col("nota_media_imdb") < 0, None)
                .otherwise(F.col("nota_media_imdb"))
        )
)

display(df_metricas_engajamento.orderBy(F.desc("nota_media_imdb")))

In [0]:
# Verificação de dados não numéricos de contagem de votos
int_check = lambda column: F.regexp(F.regexp_replace(F.col(column), r"[^\d]", ""), F.lit(r"^(\d+)$"))

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não numéricos", 
    df_metricas_engajamento,
    int_check("qtd_votos_tmdb")
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não numéricos", 
    df_metricas_engajamento,
    int_check("qtd_votos_imdb")
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não nulos", 
    df_metricas_engajamento,
    F.col("qtd_votos_tmdb").isNotNull() & F.col("qtd_votos_imdb").isNotNull()
)

df_corrupted_count = (
    df_metricas_engajamento
        .select("id_filme", "qtd_votos_imdb", "qtd_votos_tmdb")
        .where(~int_check("qtd_votos_tmdb") | ~int_check("qtd_votos_imdb"))
)

display(df_corrupted_count.limit(5))

In [0]:
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("qtd_votos_tmdb",
            F.col("qtd_votos_tmdb").try_cast("int")
        ).withColumn("qtd_votos_imdb",
            F.col("qtd_votos_imdb").try_cast("int")
        )
)

# Limpeza de valores inválidos
df_metricas_engajamento = (
    df_metricas_engajamento
        .withColumn("qtd_votos_tmdb",
            F.when(F.col("qtd_votos_tmdb") < 0, None)
                .otherwise(F.col("qtd_votos_tmdb"))
        ).withColumn("qtd_votos_imdb",
            F.when(F.col("qtd_votos_imdb") < 0, None)
                .otherwise(F.col("qtd_votos_imdb"))
        )
)

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de contagem de votos não numéricos",
    df_metricas_engajamento,
    F.col("qtd_votos_tmdb").isNotNull() & F.col("qtd_votos_imdb").isNotNull()
)

display(df_metricas_engajamento.limit(5))

In [0]:
df_metricas_engajamento.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Avaliações usuários

In [0]:
# Mapeamento tb_movies_reviews -> tb_avaliacoes_usuarios
silver_table_name = "tb_avaliacoes_usuarios"
df_avaliacoes_usuarios = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_avaliacoes_usuarios = (
    df_avaliacoes_usuarios
        .where(F.col("id").isNotNull())
        .withColumnsRenamed({
            "id": "id_filme",
            "nome": "nome_usuario",
            "nota": "nota_usuario",
            "comentario": "comentario_usuario"
        }).dropDuplicates(["id_filme","nome_usuario","nota_usuario","comentario_usuario"])
)

display(df_avaliacoes_usuarios.limit(5))

In [0]:
# Limpeza de dados de nota do usuário

is_numerical_check = F.regexp(F.col("nota_usuario"), F.lit(r"(\d+\.\d+|\d+)"))

dq_check(
    f"{silver_schema}.{silver_table_name}", 
    "Verificação de nota de usuário não numérica", 
    df_avaliacoes_usuarios,
    is_numerical_check
)

In [0]:
df_avaliacoes_usuarios = (
    df_avaliacoes_usuarios
        .withColumn("nota_usuario",
            F.when(F.col("nota_usuario") > 10, None)
                .when(F.col("nota_usuario") < 0, None)
                .otherwise(F.col("nota_usuario"))
        )
)

display(df_avaliacoes_usuarios.limit(5))

In [0]:
# Padronização dos comentários

df_avaliacoes_usuarios = (
    df_avaliacoes_usuarios
        .withColumn("comentario_usuario",
            F.regexp_replace(F.col("comentario_usuario"), r"^\s*$", "Sem comentário")
        ).withColumn("comentario_usuario",
            F.when(F.col("comentario_usuario").isNull(), "Sem comentário")
                .otherwise(F.col("comentario_usuario"))
        )
)

df_no_comment = (
    df_avaliacoes_usuarios.where(F.col("comentario_usuario") == "Sem comentário")
)

display(df_no_comment.limit(5))

In [0]:
df_avaliacoes_usuarios.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Gêneros

In [0]:
# Mapeamento tb_credits_and_tags, coluna genres -> tb_generos
silver_table_name = "tb_generos"
df_generos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_generos = (
    df_generos.select("id", "genres")
        .dropDuplicates()
        .withColumnsRenamed({
            "id": "id_filme",
            "genres": "genero"
        })
        .withColumn("genero",
            F.explode(F.split(F.upper(F.col("genero")), r"[\,\;\|]"))
        )
        .withColumn("genero",
            F.trim(F.regexp_replace(F.col("genero"), r"[^\w ]", "")))
)

display(df_generos.limit(5))

In [0]:
df_count_generos = (
    df_generos
        .groupBy("genero")
        .count()
)

display(df_count_generos)

In [0]:
# seleção dos gêneros com ao menos 200 registros

window = Window.partitionBy("genero")

df_generos = (
    df_generos
        .withColumn("count_genero", F.count(F.col("genero")).over(window))
        .where(F.col("count_genero") >= 200)
        .drop("count_genero")
)

display(df_generos.select("genero").distinct())

In [0]:
df_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")

## Pessoas Empresas

In [0]:
# Mapeamento  tb_credits_and_tags, colunas cast, directors, writers, production_companies -> tb_pessoas_empresas
silver_table_name = "tb_pessoas_empresas"
df_pessoas_empresas = spark.table(f"{bronze_schema}.tb_credits_and_tags")

df_pessoas_empresas = (
    df_pessoas_empresas.select("id", "cast", "directors", "writers", "production_companies")
        .withColumnsRenamed({
            "id": "id_filme",
        })
)

display(df_pessoas_empresas.limit(5))

In [0]:
# Melhora a limpeza dos dados de tb_pessoas_empresas (feito com ajuda de IA)

PLACEHOLDERS = ["[]", "N/A", "NA", "NENHUM", "NENHUMA", "UNKNOWN", "NONE",
                "NULL", "NAN", "NÃO INFORMADO", "-", "--"]

def clean_entity(col):
    c = F.regexp_replace(col, r'[\\"]', "")        # resíduos de \" do CSV
    c = F.regexp_replace(c, r"\s+", " ")
    return F.upper(F.trim(c))

# Lógica da blocklist: lista de termos que não devem ser considerados, e que são calculados dinamicamente
def filter_valid_entity(col, blocklist=None):
    blocked = PLACEHOLDERS + list(blocklist or [])
    return (
        col.isNotNull()
        & F.length(col).between(2, 80)       # 80+ caracteres = fragmento de sinopse, não nome
        & col.rlike(r"\p{L}")                # precisa de ao menos uma letra (descarta "1927", "59%", "9/11")
        & ~col.isin(blocked)                 # placeholders e termos de outro domínio
    )

In [0]:
# Cria blocklist com base nos valores vistos em outras colunas (feito com ajuda de IA)

generos = [
    r[0].upper() for r in \
        spark.table(f"{silver_schema}.tb_generos").select("genero").distinct().collect()
]

df_credits_and_tags = spark.table(f"{bronze_schema}.tb_credits_and_tags")
def top_tokens(c, min_count=200):
    return [r["t"] for r in (
        df_credits_and_tags.select(F.explode(F.split(F.upper(F.col(c)), r"\s*[,;|]\s*")).alias("t"))
               .groupBy("t").count().where(F.col("count") >= min_count).collect())]

blocklist = generos + top_tokens("spoken_languages") + top_tokens("production_countries")

print(blocklist)

In [0]:
# Lógica unificada para split+explode de cada tipo de entidade (feito com ajuda de IA)

def explode_entity(df, col, tipo):
    return (
        df.select("id_filme",
                  F.explode(F.split(F.col(col), r"\s*[,;|]\s*")).alias("entidade_raw"),
                  F.lit(tipo).alias("tipo_entidade"))
          .withColumn("entidade", clean_entity(F.col("entidade_raw")))
          .where(filter_valid_entity(F.col("entidade"), blocklist))
          .drop("entidade_raw")
    )

df_atores      = explode_entity(df_pessoas_empresas, "cast", "Ator")
df_diretores   = explode_entity(df_pessoas_empresas, "directors", "Diretor")
df_roteiristas = explode_entity(df_pessoas_empresas, "writers", "Roteirista")
df_produtoras  = explode_entity(df_pessoas_empresas, "production_companies", "Produtora")

In [0]:
df_pessoas_empresas = (
    df_atores
        .union(df_diretores)
        .union(df_roteiristas)
        .union(df_produtoras)
        .dropDuplicates()
)

display(df_pessoas_empresas.groupBy("tipo_entidade").count())

In [0]:
(
    df_pessoas_empresas
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{silver_table_name}")
)

## Cotação Dólar

In [0]:
# Mapeamento tb_cotacao_dolar -> tb_cotacao_dolar
silver_table_name = "tb_cotacao_dolar"
df_cotacao_dolar = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

df_cotacao_dolar = (
    df_cotacao_dolar
        .withColumnRenamed("cotacaoCompra", "cotacao_compra")
        .withColumnRenamed("cotacaoVenda", "cotacao_venda")
        .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
        .drop("dataHoraCotacao")
        .dropDuplicates(["data_cotacao"])
        .orderBy("data_cotacao")
)

display(df_cotacao_dolar.limit(7))

In [0]:
# Reindexa a tabela com todos os dias entre a primeira data e hoje
min_date = df_cotacao_dolar.select(F.min("data_cotacao")).collect()[0][0]
today = datetime.today().strftime("%Y-%m-%d")

new_idx = pd.date_range(start=min_date, end=today, freq="D")

pandas_df = df_cotacao_dolar.toPandas()

pandas_df = (
    pandas_df.set_index('data_cotacao')
        .sort_index()
        .reindex(new_idx, method="pad")
        .reset_index(names="data_cotacao")
)

df_cotacao_dolar = (
    spark.createDataFrame(pandas_df)
        .withColumn("data_cotacao",
            F.to_date(F.col("data_cotacao"), "yyyy-MM-dd")
        )
)

display(df_cotacao_dolar.limit(7))

In [0]:
df_cotacao_dolar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.{silver_table_name}")